# AS TWAS + COLOC Pipeline: Mock Walkthrough

**Project**: Towards Coloc-Confirmed TWAS of Ankylosing Spondylitis

This notebook demonstrates the pipeline in **mock/demo mode**. No external data, GTEx models, MetaXcan, or R installation is required.

For real execution, see the prerequisites in `README.md`.

---

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print('Dependencies loaded successfully.')

## 1. Load Configuration

The pipeline is fully configuration-driven via `config/as.yaml`.

In [ ]:
from src.config import load_config, get_coloc_priors, get_tissues

cfg = load_config('../config/as.yaml')
print('Project:', cfg['project']['name'])
print('Disease:', cfg['project']['disease'])
print('Target build:', cfg['project']['genome_build'])
print('\nColoc priors:', get_coloc_priors(cfg))
print('\nTissues configured:', get_tissues(cfg))

## 2. GWAS Harmonization

The harmonization stage:
- Renames raw columns to canonical schema
- Constructs GTEx-style varIDs (`chr{chrom}_{pos}_{ref}_{alt}_b38`)
- Drops ambiguous palindromic SNPs (AT/CG)
- Validates the required schema

**Key pitfall**: If coordinates are on hg19, a liftover to hg38 is required before varID construction. Without this, S-PrediXcan will report 0% of model SNPs found.

In [ ]:
from src.harmonization import harmonize_gwas

# Mock raw GWAS data
rng = np.random.default_rng(42)
n = 50
_a1 = ['A', 'C', 'G', 'T']
_a2 = ['C', 'A', 'T', 'C']  # non-palindromic pairs
raw_gwas = pd.DataFrame({
    'CHR':  [str(i % 22 + 1) for i in range(n)],
    'BP':   rng.integers(1_000_000, 200_000_000, n),
    'A1':   [_a1[i % 4] for i in range(n)],
    'A2':   [_a2[i % 4] for i in range(n)],
    'BETA': rng.normal(0, 0.05, n),
    'SE':   rng.uniform(0.01, 0.05, n),
    'P':    rng.uniform(0, 1, n),
    'N':    [10619] * n,
})
print('Raw GWAS shape:', raw_gwas.shape)
raw_gwas.head(3)

In [ ]:
harmonized = harmonize_gwas(raw_gwas, cfg)
print('Harmonized GWAS shape:', harmonized.shape)
print('\nSample varIDs:')
print(harmonized['varID'].head(5).tolist())
harmonized[['varID', 'chrom', 'pos', 'effect_allele', 'non_effect_allele', 'beta', 'se', 'pvalue']].head()

## 3. S-PrediXcan Command Building

In real mode, these commands are executed for each tissue. In mock mode, we inspect the command without running it.

**Critical**: `--snp_column` must be `varID` (not `rsID`) to match GTEx PredictDB models.

In [ ]:
from src.external_tools.spredixcan_wrapper import build_spredixcan_command

cmd = build_spredixcan_command(
    cfg=cfg,
    tissue='Whole_Blood',
    model_db='data/reference/gtex_v8_models/gtex_v8_mashr_Whole_Blood.db',
    covariance='data/reference/gtex_v8_models/gtex_v8_mashr_Whole_Blood.txt.gz',
    gwas_file='data/interim/gwas_harmonized/as_gwas_hg38_varid.tsv.gz',
    output_file='results/twas/Whole_Blood.spredixcan.csv',
)
print('S-PrediXcan command (would be run in real mode):')
print('\n  '.join(cmd))

## 4. Mock TWAS Results

Synthetic S-PrediXcan results across multiple tissues, followed by global BH/FDR correction.

In [ ]:
from src.twas import make_mock_spredixcan_result, apply_fdr_correction, filter_twas_hits

tissues = get_tissues(cfg)
frames = [make_mock_spredixcan_result(t, seed=i) for i, t in enumerate(tissues)]
twas_all = pd.concat(frames, ignore_index=True)
print(f'Total gene-tissue pairs: {len(twas_all)}')

# Apply FDR correction
twas_all = apply_fdr_correction(twas_all, alpha=0.05)
twas_hits = filter_twas_hits(twas_all)
print(f'TWAS-significant hits (FDR q < 0.05): {len(twas_hits)}')
twas_hits[['gene', 'gene_name', 'tissue', 'zscore', 'pvalue', 'pvalue_adj']].head(5)

## 5. COLOC: varbeta computation

**Critical**: `coloc.abf` requires `varbeta = se**2`, NOT `se` directly. This is one of the most common coloc mistakes.

In [ ]:
from src.coloc import compute_varbeta, make_mock_coloc_result, interpret_coloc_result

# Demonstrate varbeta computation
example_df = pd.DataFrame({'se': [0.05, 0.10, 0.20], 'beta': [0.1, -0.2, 0.05]})
result_df = compute_varbeta(example_df)
print('varbeta = se^2:')
print(result_df[['se', 'varbeta']])

In [ ]:
# Mock coloc result interpretation
raw_result = make_mock_coloc_result(pp4=0.85, pp3=0.10)
annotated = interpret_coloc_result(raw_result, cfg)

print('PP4:        ', round(annotated['PP.H4.abf'], 3))
print('PP3+PP4:    ', round(annotated['PP3_PP4'], 3))
print('PP4 ratio:  ', round(annotated['PP4_ratio'], 3))
print('Coloc hit:  ', annotated['coloc_hit'])
print('Powered:    ', annotated['powered_locus'])
print('Strong:     ', annotated['strong_shared'])

## 6. Visualization

All plots use mock data in demo mode. Real data will flow through the same functions.

In [ ]:
from src.visualization import (
    make_mock_twas_df, make_mock_coloc_df,
    plot_twas_qqplot, plot_coloc_pp4_barplot, plot_coloc_pp4_vs_pp3pp4
)
import io

mock_twas = make_mock_twas_df(n=200)
mock_coloc = make_mock_coloc_df(n=50)

# QQ-plot
plot_twas_qqplot(
    pvalues=mock_twas['pvalue'],
    output_path='/tmp/notebook_qqplot.png',
    dpi=100,
)
from IPython.display import Image
Image('/tmp/notebook_qqplot.png')

In [ ]:
# COLOC PP4 barplot
plot_coloc_pp4_barplot(
    coloc_df=mock_coloc,
    output_path='/tmp/notebook_coloc_barplot.png',
    top_n=15,
    dpi=100,
)
Image('/tmp/notebook_coloc_barplot.png')

In [ ]:
# PP4 vs PP3+PP4 scatter
plot_coloc_pp4_vs_pp3pp4(
    coloc_df=mock_coloc,
    output_path='/tmp/notebook_pp4_scatter.png',
    dpi=100,
)
Image('/tmp/notebook_pp4_scatter.png')

## 7. Full Mock Pipeline Run

Run all 14 pipeline stages end-to-end using mock data.

In [ ]:
import tempfile, shutil
from src.pipeline import run_pipeline

tmpdir = tempfile.mkdtemp(prefix='as_twas_nb_')
print(f'Output directory: {tmpdir}')

summary = run_pipeline(
    config_path='../config/as.yaml',
    mock=True,
    base_dir=tmpdir,
)

print('\n── Pipeline Summary ──────────────────')
for k, v in summary.items():
    print(f'  {k:<30} {v}')

In [ ]:
# Show top-genes table
from pathlib import Path
top_genes_path = Path(tmpdir) / 'results' / 'tables' / 'top_genes_pp4.tsv'
if top_genes_path.exists():
    top_genes = pd.read_csv(top_genes_path, sep='\t')
    print(f'Top coloc-confirmed genes (PP4 >= 0.7): {len(top_genes)}')
    display(top_genes)
else:
    print('No coloc hits in mock run (random data — expected).')

In [ ]:
# Show manifest
import json, glob
manifests = glob.glob(str(Path(tmpdir) / 'results' / 'manifests' / '*.json'))
if manifests:
    with open(manifests[0]) as f:
        manifest = json.load(f)
    print('Manifest keys:', list(manifest.keys()))
    print('Timestamp:    ', manifest['timestamp'])
    print('Coloc priors: ', manifest['coloc_priors'])

In [ ]:
# Clean up
shutil.rmtree(tmpdir)
print('Temporary files cleaned up.')

---

## Next Steps for Real Analysis

1. **Download AS GWAS summary stats** → `data/raw/as_gwas.tsv.gz`
2. **Download GTEx v8 PredictDB models** → `data/reference/gtex_v8_models/`
3. **Install MetaXcan** → `external/MetaXcan/`
4. **Install R + coloc** → update `config/as.yaml`
5. **Run**: `python scripts/run_pipeline.py --no-mock`
6. **Update** `docs/manuscript/manuscript.tex` Results section with real findings

See `README.md` for full setup instructions and harmonization pitfalls.